In [ ]:
!pip install tensorflow moviepy opencv-python
!pip install numpy
!pip install openai
! pip3 install --upgrade --user google-cloud-aiplatform

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import cv2
import numpy as np
from moviepy.editor import VideoFileClip
from google.colab.patches import cv2_imshow
import openai
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import os
from vertexai.generative_models import GenerationConfig, GenerativeModel, Image, Part


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
video_path = ("/content/drive/My Drive/Como criar uma CÂMERA em TERCEIRA PESSOA na Unity SEM PROGRAMAR usando Cinemachine (720p60).mp4")

In [ ]:


def load_and_preprocess_video(video_path):
    try:
        video = VideoFileClip(video_path)
    except Exception as e:
        print(f"Erro ao carregar o vídeo: {e}")
        return None

    frames = []
    for frame in video.iter_frames():
        frame = cv2.resize(frame, (64, 64))
        frame = frame / 255.0
        frames.append(frame)
    return np.array(frames)

video_path = video_path
frames = load_and_preprocess_video(video_path)

if frames is not None:
    print(f"Total de frames processados: {len(frames)}")
else:
    print("Falha ao processar os frames.")


In [ ]:
def create_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = create_model()

In [ ]:
num_frames = 1000
num_classes = 10

labels = np.random.randint(0, num_classes, num_frames)

frames = np.random.rand(num_frames, 64, 64, 3)



In [ ]:
def create_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = create_model()

In [ ]:
model.fit(frames, labels, epochs=10, batch_size=32)


In [ ]:
def predict_on_video(model, video_path):
    frames = load_and_preprocess_video(video_path)
    if frames is not None:
        predictions = model.predict(frames)
        return predictions
    else:
        print("Falha ao carregar e processar o vídeo.")
        return None

video_path = video_path
predictions = predict_on_video(model, video_path)

if predictions is not None:
    for i, prediction in enumerate(predictions):
        print(f"Frame {i}: Classe {np.argmax(prediction)}")


In [ ]:

def load_classes(file_path):
    with open(file_path, 'r') as f:
        classes = f.read().strip().split('\n')
    return classes

cfg_path = '/content/drive/My Drive/yolov3.cfg'
weights_path = '/content/drive/My Drive/yolov3.weights'
names_path = '/content/drive/My Drive/coco.names'

net = cv2.dnn.readNet(weights_path, cfg_path)
classes = load_classes(names_path)

def detect_objects(frame, net, output_layers):
    height, width = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
    net.setInput(blob)
    outs = net.forward(output_layers)

    class_ids = []
    confidences = []
    boxes = []

    for out in outs:
        for detection in out:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]

            if confidence > 0.5:

                center_x = int(detection[0] * width)
                center_y = int(detection[1] * height)
                w = int(detection[2] * width)
                h = int(detection[3] * height)
                x = int(center_x - w / 2)
                y = int(center_y - h / 2)

                boxes.append([x, y, w, h])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

    result = []
    for i in range(len(boxes)):
        if i in indexes:
            x, y, w, h = boxes[i]
            label = str(classes[class_ids[i]])
            result.append((x, y, w, h, label))

    return result

def draw_boxes(frame, detections):
    for (x, y, w, h, label) in detections:
        color = (0, 255, 0)
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
        cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

video_path = '/content/drive/My Drive/Como criar uma CÂMERA em TERCEIRA PESSOA na Unity SEM PROGRAMAR usando Cinemachine (720p60).mp4'
cap = cv2.VideoCapture(video_path)

layer_names = net.getLayerNames()
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

all_detections = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    detections = detect_objects(frame, net, output_layers)
    draw_boxes(frame, detections)
    all_detections.append(detections)

    # cv2_imshow(frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [33]:
!pip -q install google-generativeai==0.3.0
!pip -q install google-ai-generativelanguage==0.4.0

In [ ]:
# setup gemini
!pip install openai

In [47]:
from openai import OpenAI
import os
import json


client = OpenAI(
    # This is the default and can be omitted
    api_key=userdata.get('OPENAI_KEY'),
)


In [ ]:
def generate_description(detections, frame_number):
    objects = [d[4] for d in detections]
    prompt = f"No frame {frame_number}, os seguintes objetos foram detectados: {', '.join(objects)}. Descreva a cena."
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Você é um assistente útil."},
            {"role": "user", "content": prompt}
        ]
    )
    return response['choices'][0]['message']['content'].strip()

# Inicializar uma lista para armazenar as descrições
all_descriptions = []

# Processar os resultados de detecção de objetos e gerar descrições
for frame_number, detections in enumerate(all_detections):
    description = generate_description(detections, frame_number)
    all_descriptions.append((frame_number, description))

# Exibir as descrições geradas
for frame_number, description in all_descriptions:
    print(f"Frame {frame_number}: {description}")

# Função para fazer perguntas sobre o vídeo
def ask_question(question):
    context = "\n".join([f"Frame {fn}: {desc}" for fn, desc in all_descriptions])
    prompt = f"Aqui estão as descrições dos frames do vídeo:\n{context}\n\nPergunta: {question}\nResposta:"
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "Você é um assistente útil."},
            {"role": "user", "content": prompt}
        ]
    )
    return response['choices'][0]['message']['content'].strip()

# Exemplo de uso da função de perguntas
question = "Quantos carros foram detectados no vídeo?"
answer = ask_question(question)
print(f"Pergunta: {question}\nResposta: {answer}")